# Lab 5 — Evaluating a RAG Agent

Building a grounded agent is half the job — **proving** it behaves correctly is the other half. This lab evaluates the RAG agent from **Lab 4** against a small, hand-written test set and scores three behaviours that matter for any internal knowledge assistant:

| Metric | Question it answers |
|---|---|
| **Groundedness** | Did the answer come from the retrieved documents (not invented)? |
| **Citation** | Did the answer cite a source? |
| **Out-of-scope refusal** | Did the agent refuse questions outside the knowledge base? |

You'll run two styles of evaluation:

1. **Deterministic checks** (no LLM) — canary-token grounding, citation presence, refusal detection.
2. **LLM-as-judge** — an LLM scores correctness/groundedness for nuanced cases.

> **Prerequisites:** the Lab 4 agent (`labs-hosted-agent-rag`) deployed and reachable, with the seeded `contoso-outdoors` index. Set the same `.env` values as Lab 4.

## 1. Define the evaluation set

Each test case has a question, the **canary token** we expect a grounded answer to contain (or `None` for out-of-scope cases), and whether the agent should **refuse**. Swap these for questions over your own knowledge base.

In [ ]:
TEST_CASES = [
    # In-scope: a grounded answer must contain the canary token and a citation.
    {"question": "What is your return policy? Include any item codes.",
     "expect_canary": "TR-CANARY-7821", "should_refuse": False},
    {"question": "How long does standard shipping take and is there a promo code?",
     "expect_canary": "SHIP-CANARY-4493", "should_refuse": False},
    {"question": "How do I clean my tent and what SKU is the re-waterproofing kit?",
     "expect_canary": "TENT-CANARY-9067", "should_refuse": False},
    # Out-of-scope: the agent should refuse (no canary expected).
    {"question": "What is the capital of France?",
     "expect_canary": None, "should_refuse": True},
    {"question": "Write me a poem about the stock market.",
     "expect_canary": None, "should_refuse": True},
]
print(f"{len(TEST_CASES)} test cases ready.")

## 2. Collect agent responses

We call the deployed Lab 4 agent over the Responses protocol for each question and capture the text output.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv()

AGENT_NAME = "labs-hosted-agent-rag"
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
credential = DefaultAzureCredential()
url = f"{project_endpoint}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1"


def ask_agent(question: str) -> str:
    token = credential.get_token("https://ai.azure.com/.default").token
    resp = requests.post(
        url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"input": question, "store": False},
        timeout=120,
    )
    data = resp.json()
    texts = []
    for item in data.get("output", []):
        for part in item.get("content", []):
            if part.get("type") == "output_text":
                texts.append(part["text"])
    return "\n".join(texts).strip()


for case in TEST_CASES:
    case["answer"] = ask_agent(case["question"])
    print(f"Q: {case['question']}\nA: {case['answer'][:160]}...\n")

## 3. Deterministic evaluation (no LLM)

Fast, cheap, and reproducible checks:

- **Grounded** — for in-scope cases, the expected canary token appears in the answer.
- **Cited** — the answer references a source (a URL or the word "source").
- **Refused** — for out-of-scope cases, the answer signals refusal and contains **no** canary token.

In [ ]:
import re

REFUSAL_MARKERS = (
    "don't have", "do not have", "can't help", "cannot help", "out of scope",
    "outside", "not able to", "no information", "unable to", "not covered",
)


def looks_refused(text: str) -> bool:
    low = text.lower()
    return any(m in low for m in REFUSAL_MARKERS)


def is_cited(text: str) -> bool:
    return bool(re.search(r"https?://|source", text, re.IGNORECASE))


rows = []
for case in TEST_CASES:
    ans = case["answer"]
    if case["should_refuse"]:
        passed = looks_refused(ans) and "CANARY" not in ans
        detail = "refused" if looks_refused(ans) else "did NOT refuse"
    else:
        grounded = case["expect_canary"] in ans
        cited = is_cited(ans)
        passed = grounded and cited
        detail = f"grounded={grounded}, cited={cited}"
    rows.append((case["question"][:45], "PASS" if passed else "FAIL", detail))

print(f"{'Question':47}{'Result':8}Detail")
print("-" * 80)
for q, res, detail in rows:
    print(f"{q:47}{res:8}{detail}")

passed = sum(1 for _, r, _ in rows if r == "PASS")
print(f"\nDeterministic pass rate: {passed}/{len(rows)} ({passed / len(rows):.0%})")

## 4. LLM-as-judge (groundedness & correctness)

Deterministic checks catch the obvious cases; an LLM judge handles nuance (paraphrasing, partial answers). We ask a model to score each in-scope answer on a 1–5 scale for how well it is **grounded in and consistent with** the retrieved source content.

This is the same idea as a *correctness judge* in a RAG benchmark — useful when comparing two implementations (e.g. custom RAG vs a low-code tool).

In [ ]:
from azure.ai.projects import AIProjectClient

model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1")
project = AIProjectClient(endpoint=project_endpoint, credential=credential)
judge = project.get_openai_client()

JUDGE_PROMPT = (
    "You are an evaluation judge for a knowledge-base assistant. "
    "Given a QUESTION and the assistant's ANSWER, rate from 1 to 5 how well the answer is "
    "grounded, specific, and helpful. Reply with ONLY the integer score."
)

scores = []
for case in TEST_CASES:
    if case["should_refuse"]:
        continue  # judge only in-scope answers
    result = judge.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"QUESTION: {case['question']}\nANSWER: {case['answer']}"},
        ],
    )
    raw = result.choices[0].message.content.strip()
    match = re.search(r"[1-5]", raw)
    score = int(match.group()) if match else 0
    scores.append(score)
    print(f"score={score}  Q: {case['question'][:50]}")

if scores:
    print(f"\nAverage LLM judge score: {sum(scores) / len(scores):.2f} / 5")

## 5. Summary

You now have a repeatable, two-layer evaluation harness:

- **Deterministic** grounding / citation / refusal checks — cheap regression gates you can run in CI.
- **LLM-as-judge** correctness — for nuanced scoring and comparing implementations.

**Extend it for your own knowledge base:**

- Replace `TEST_CASES` with real questions and expected facts from your documents.
- Add metrics such as **embedding similarity** between answer and source chunk, or **latency** per query.
- Track scores over time (each model or prompt change) to catch regressions before release.
- Use the same harness to benchmark a custom RAG agent against a low-code alternative.